# Apache Iceberg Lakehouse — Student Notebook

1. **Setup** — configuration, S3 Tables catalog, upload scripts and data
2. **Bronze** — run Glue jobs, validate with Athena
3. **Silver** — run Glue job, validate with Athena
4. **Gold** — run Glue job, validate with Athena
5. **Iceberg Features** — snapshots, time-travel

---

# Part 1: Setup

## Step 1: Configuration

Set up all resource names matching the CloudFormation stack outputs. Bronze uses a
standard S3 bucket with Glue Catalog Iceberg tables. Silver and Gold use **S3 Tables**
(managed Iceberg) accessed through the `s3tablescatalog` federated catalog.

Fill in the values from your CloudFormation stack outputs.

In [1]:
import os, time
import boto3
import pandas as pd

AWS_REGION = 'us-east-1'

# Account ID comes from the authenticated session (never hardcoded, no keys in this notebook)
ACCOUNT_ID = boto3.client('sts', region_name=AWS_REGION).get_caller_identity()['Account']

# Resource names follow the lab's naming convention. VERIFY them against the AWS
# Console (S3, Athena, S3 Tables) - the checks at the bottom fail fast if they are wrong.
BUCKET = f'ecommerce-analytics-datalake-dev-{ACCOUNT_ID}'              # S3 data lake bucket
DATABASE = 'ecommerce_analytics_bronze_dev'
ATHENA_WORKGROUP = 'ecommerce-analytics-workgroup-dev'                  # Athena workgroup

# S3 Tables catalog (silver + gold layers)
TABLE_BUCKET_NAME = f'ecommerce-analytics-gold-dev-{ACCOUNT_ID}'       # S3 Table Bucket
S3T_CATALOG = f's3tablescatalog/{TABLE_BUCKET_NAME}'
SILVER_DB = 'ecommerce_silver'
GOLD_DB = 'ecommerce_gold'

# Glue job names
CDC_JOB = 'ecommerce-analytics-cdc-bronze-dev'
EVENTS_JOB = 'ecommerce-analytics-events-bronze-dev'
SILVER_JOB = 'ecommerce-analytics-silver-layer-dev'
GOLD_JOB = 'ecommerce-analytics-gold-layer-dev'

s3 = boto3.client('s3', region_name=AWS_REGION)
glue = boto3.client('glue', region_name=AWS_REGION)
athena = boto3.client('athena', region_name=AWS_REGION)
s3tables = boto3.client('s3tables', region_name=AWS_REGION)

# Fail fast if any configured resource does not exist
s3.head_bucket(Bucket=BUCKET)
athena.get_work_group(WorkGroup=ATHENA_WORKGROUP)
s3tables.get_table_bucket(tableBucketARN=f'arn:aws:s3tables:{AWS_REGION}:{ACCOUNT_ID}:bucket/{TABLE_BUCKET_NAME}')
for _job in (CDC_JOB, EVENTS_JOB, SILVER_JOB, GOLD_JOB):
    glue.get_job(JobName=_job)

print(f'\u2705 Configuration verified (account {ACCOUNT_ID}, region {AWS_REGION})')
print(f'   bucket={BUCKET}\n   workgroup={ATHENA_WORKGROUP}\n   table bucket={TABLE_BUCKET_NAME}')

✅ Configuration verified (account 124435207106, region us-east-1)
   bucket=ecommerce-analytics-datalake-dev-124435207106
   workgroup=ecommerce-analytics-workgroup-dev
   table bucket=ecommerce-analytics-gold-dev-124435207106


## Step 2: Register S3 Tables Catalog (one-time after stack deploy)

The Silver and Gold layers write to **S3 Tables** (managed Iceberg). To query these
tables from Athena and allow Glue jobs to discover them, we must register a
`s3tablescatalog` federated catalog in the Glue Data Catalog. This bridges the
S3 Tables API with AWS analytics services (Athena, Glue, EMR). The catalog uses
`IAM_ALLOWED_PRINCIPALS` so access is controlled by IAM policies, not Lake Formation grants.

This only needs to run once per account/region — it's safe to re-run (idempotent).

In [2]:
try:
    glue.create_catalog(
        Name='s3tablescatalog',
        CatalogInput={
            'FederatedCatalog': {
                'Identifier': f'arn:aws:s3tables:{AWS_REGION}:{ACCOUNT_ID}:bucket/*',
                'ConnectionName': 'aws:s3tables'
            },
            'CreateDatabaseDefaultPermissions': [
                {'Principal': {'DataLakePrincipalIdentifier': 'IAM_ALLOWED_PRINCIPALS'}, 'Permissions': ['ALL']}
            ],
            'CreateTableDefaultPermissions': [
                {'Principal': {'DataLakePrincipalIdentifier': 'IAM_ALLOWED_PRINCIPALS'}, 'Permissions': ['ALL']}
            ],
            'AllowFullTableExternalDataAccess': 'True'
        }
    )
    print('\u2705 s3tablescatalog registered')
except glue.exceptions.AlreadyExistsException:
    print('\u2705 s3tablescatalog already exists')

# Verify the registration really exists (do not trust the absence of an exception)
cat = glue.get_catalog(CatalogId='s3tablescatalog')['Catalog']
print('Verified catalog:', cat['Name'], '->', cat.get('FederatedCatalog', {}).get('Identifier'))

✅ s3tablescatalog already exists
Verified catalog: s3tablescatalog -> arn:aws:s3tables:us-east-1:124435207106:bucket/*


## Step 3: Upload Glue Scripts and Datasets

Upload the four Glue ETL scripts and the CDC/events datasets to S3. The Glue jobs
fetch their scripts from `s3://{bucket}/glue-scripts/` at runtime, so this must
happen before running any jobs.

In [3]:
import glob as globmod

# Upload Glue scripts
scripts = {
    'glue-jobs/cdc-batch/src/iceberg-cdc-bronze-layer.py': 'glue-scripts/iceberg-cdc-bronze-layer.py',
    'glue-jobs/events/src/iceberg-events-bronze-layer.py': 'glue-scripts/iceberg-events-bronze-layer.py',
    'glue-jobs/silver/src/iceberg-silver-layer.py': 'glue-scripts/iceberg-silver-layer.py',
    'glue-jobs/gold/src/iceberg-gold-layer.py': 'glue-scripts/iceberg-gold-layer.py',
}
for local, s3_key in scripts.items():
    s3.upload_file(local, BUCKET, s3_key)
    print(f'\u2705 Uploaded {s3_key}')

# Upload datasets
for local_path in globmod.glob('datasets/**/*', recursive=True):
    if os.path.isfile(local_path):
        s3.upload_file(local_path, BUCKET, local_path)

resp = s3.list_objects_v2(Bucket=BUCKET, Prefix='datasets/', MaxKeys=5)
for obj in resp.get('Contents', []):
    print(f'  {obj["Key"]} ({obj["Size"]:,} bytes)')
print('\u2705 Datasets uploaded')

✅ Uploaded glue-scripts/iceberg-cdc-bronze-layer.py
✅ Uploaded glue-scripts/iceberg-events-bronze-layer.py
✅ Uploaded glue-scripts/iceberg-silver-layer.py
✅ Uploaded glue-scripts/iceberg-gold-layer.py
  datasets/cdc_data/small/year=2025/table_name=orders/month=01/data_89a7daa5.json (154,082 bytes)
  datasets/cdc_data/small/year=2025/table_name=orders/month=02/data_45393fc2.json (178,709 bytes)
  datasets/cdc_data/small/year=2025/table_name=orders/month=03/data_d44f1e66.json (179,452 bytes)
  datasets/cdc_data/small/year=2025/table_name=orders/month=04/data_1c051040.json (177,904 bytes)
  datasets/cdc_data/small/year=2025/table_name=orders/month=05/data_88f6016a.json (177,918 bytes)
✅ Datasets uploaded


## Step 4: Helper Functions

`run_glue_job` starts a Glue job and polls until completion. `run_query` executes
Athena SQL and returns a DataFrame. The optional `catalog` and `database` parameters
let us query S3 Tables (silver/gold) by passing the federated catalog name.

In [4]:
RUNS = {}   # evidence: job name -> {run_id, state, seconds}

def run_glue_job(job_name, arguments=None, poll_seconds=30):
    """Start a Glue job, poll to a terminal state, record the evidence, return the state."""
    kwargs = {'JobName': job_name}
    if arguments:
        kwargs['Arguments'] = arguments
    run_id = glue.start_job_run(**kwargs)['JobRunId']
    print(f'\u25b6 Started {job_name} (run: {run_id})')
    while True:
        status = glue.get_job_run(JobName=job_name, RunId=run_id)['JobRun']
        state = status['JobRunState']
        if state in ('SUCCEEDED', 'FAILED', 'STOPPED', 'ERROR', 'TIMEOUT'):
            break
        print(f'  {state}...', end='\r')
        time.sleep(poll_seconds)
    RUNS[job_name] = {'run_id': run_id, 'state': state, 'seconds': status.get('ExecutionTime', 0)}
    if state == 'SUCCEEDED':
        print(f'\u2705 {job_name} SUCCEEDED ({status.get("ExecutionTime", 0)}s, run {run_id})')
    else:
        print(f'\u274c {job_name} {state}: {status.get("ErrorMessage", "")}')
        print(f'   Logs: CloudWatch /aws-glue/jobs/error, stream {run_id}')
    return state

def run_query(sql, catalog=None, database=None):
    params = {
        'QueryString': sql,
        'WorkGroup': ATHENA_WORKGROUP,
        'ResultConfiguration': {'OutputLocation': f's3://{BUCKET}/athena-results/'}
    }
    if catalog or database:
        params['QueryExecutionContext'] = {}
        if catalog:
            params['QueryExecutionContext']['Catalog'] = catalog
        if database:
            params['QueryExecutionContext']['Database'] = database
    qid = athena.start_query_execution(**params)['QueryExecutionId']
    while True:
        status = athena.get_query_execution(QueryExecutionId=qid)
        state = status['QueryExecution']['Status']['State']
        if state in ('SUCCEEDED', 'FAILED', 'CANCELLED'):
            break
        time.sleep(1)
    if state != 'SUCCEEDED':
        raise Exception(status['QueryExecution']['Status'].get('StateChangeReason', state))
    # Paginate: get_query_results returns at most 1000 rows per page
    cols, rows, first = None, [], True
    for page in athena.get_paginator('get_query_results').paginate(QueryExecutionId=qid):
        if cols is None:
            cols = [c['Label'] for c in page['ResultSet']['ResultSetMetadata']['ColumnInfo']]
        page_rows = page['ResultSet']['Rows']
        if first:
            page_rows = page_rows[1:]   # first row of first page is the header
            first = False
        rows += [[f.get('VarCharValue') for f in r['Data']] for r in page_rows]
    return pd.DataFrame(rows, columns=cols)

print('\u2705 Helpers ready')

✅ Helpers ready


---
# Part 2: Bronze Layer

## Step 5: Run Bronze Glue Jobs

Two jobs ingest raw data into Iceberg tables on S3:
- **CDC Bronze** — reads Debezium CDC JSON, deduplicates by primary key, and uses
  `MERGE INTO` to upsert orders, users, and products.
- **Events Bronze** — reads clickstream JSON and appends to the events table.

In [5]:
run_glue_job(CDC_JOB)
run_glue_job(EVENTS_JOB)

▶ Started ecommerce-analytics-cdc-bronze-dev (run: jr_764188bd21bc9c6f0c71ccb4f8cde48064d086b3fd845731c3569621c1d35791)
✅ ecommerce-analytics-cdc-bronze-dev SUCCEEDED (175s, run jr_764188bd21bc9c6f0c71ccb4f8cde48064d086b3fd845731c3569621c1d35791)
▶ Started ecommerce-analytics-events-bronze-dev (run: jr_dd5b37ae54998ad98bcc09806e22fc31f592964e8eba6e0bb73fa8dc14276d44)
✅ ecommerce-analytics-events-bronze-dev SUCCEEDED (84s, run jr_dd5b37ae54998ad98bcc09806e22fc31f592964e8eba6e0bb73fa8dc14276d44)


'SUCCEEDED'

## Step 6: Validate Bronze Layer

Write Athena queries to validate the bronze tables:
1. Count rows in each table (orders, users, products, events)
2. Check for duplicate order IDs (should be 0 after MERGE dedup)
3. Show event type distribution

Use `run_query()` — bronze tables are in the default Glue catalog, so no catalog parameter needed.
Reference tables as `{DATABASE}.table_name`.

In [6]:
bronze_counts = run_query(f"""
    SELECT 'orders' AS table_name, COUNT(*) AS row_count FROM {DATABASE}.orders
    UNION ALL SELECT 'users', COUNT(*) FROM {DATABASE}.users
    UNION ALL SELECT 'products', COUNT(*) FROM {DATABASE}.products
    UNION ALL SELECT 'events', COUNT(*) FROM {DATABASE}.events
""")
print(bronze_counts.to_string(index=False))

table_name row_count
     users       912
  products       393
    events       996
    orders      2133


In [7]:
dupes = run_query(f"SELECT order_id, COUNT(*) AS duplicate_count FROM {DATABASE}.orders GROUP BY order_id HAVING COUNT(*) > 1")
print(f'Duplicate orders: {len(dupes)}')
assert len(dupes) == 0, 'MERGE dedup failed: duplicate order_ids found'

Duplicate orders: 0


In [8]:
event_types = run_query(f"SELECT event_type, COUNT(*) AS cnt FROM {DATABASE}.events GROUP BY event_type ORDER BY cnt DESC")
print(event_types.to_string(index=False))

 event_type cnt
  page_view 416
add_to_cart 240
   purchase 186
      click 154


---
# Part 3: Silver Layer

## Step 7: Run Silver Glue Job

Reads from Bronze (Glue Catalog / S3) and writes enriched tables to **S3 Tables**:
- **order_details** — orders joined with users, adds customer segments and lifetime value
- **enriched_events** — events joined with users and products, adds revenue and session sequence
- **product_performance** — aggregated metrics per product (views, carts, purchases, conversion rates)

In [9]:
run_glue_job(SILVER_JOB)

▶ Started ecommerce-analytics-silver-layer-dev (run: jr_b369d14c8b0b4f9f0df3b34d38c9873601054ceb15f9c07603571e15d6b6b2c1)
✅ ecommerce-analytics-silver-layer-dev SUCCEEDED (95s, run jr_b369d14c8b0b4f9f0df3b34d38c9873601054ceb15f9c07603571e15d6b6b2c1)


'SUCCEEDED'

## Step 8: Validate Silver Layer

Write Athena queries to validate the silver tables:
1. Count rows in each table (order_details, enriched_events, product_performance)
2. Show customer segment distribution from order_details
3. Show product funnel conversion rates from product_performance

Silver tables are on **S3 Tables** — pass `catalog=S3T_CATALOG, database=SILVER_DB`
to `run_query()`. Use unqualified table names (no database prefix in SQL).

In [10]:
silver_counts = run_query("""
    SELECT 'order_details' AS table_name, COUNT(*) AS row_count FROM order_details
    UNION ALL SELECT 'enriched_events', COUNT(*) FROM enriched_events
    UNION ALL SELECT 'product_performance', COUNT(*) FROM product_performance
""", catalog=S3T_CATALOG, database=SILVER_DB)
print(silver_counts.to_string(index=False))

         table_name row_count
      order_details      2024
    enriched_events       996
product_performance       393


In [11]:
segments = run_query("""
    SELECT customer_segment, COUNT(*) AS orders, ROUND(AVG(total_amount), 2) AS avg_order_value
    FROM order_details
    GROUP BY customer_segment
    ORDER BY orders DESC
""", catalog=S3T_CATALOG, database=SILVER_DB)
print(segments.to_string(index=False))

customer_segment orders avg_order_value
           Loyal   1076          261.31
          Repeat    516          258.75
             VIP    217          262.23
             New    215          265.97


In [12]:
funnel = run_query("""
    SELECT category,
           SUM(page_views) AS page_views,
           SUM(add_to_carts) AS add_to_carts,
           SUM(purchases) AS purchases,
           COALESCE(ROUND(100.0 * SUM(purchases) / NULLIF(SUM(page_views), 0), 2), 0.0) AS view_to_purchase_rate_pct
    FROM product_performance
    GROUP BY category
    ORDER BY page_views DESC
""", catalog=S3T_CATALOG, database=SILVER_DB)
print(funnel.to_string(index=False))

     category page_views add_to_carts purchases view_to_purchase_rate_pct
       Sports        120           72        48                      40.0
  Electronics        100           42        28                      28.0
Home & Garden         50           20        24                      48.0
        Books         36           22        16                     44.44
     Clothing          0            0         0                       0.0


---
# Part 4: Gold Layer

## Step 9: Run Gold Glue Job

Reads from Silver (S3 Tables) and writes business-ready aggregations to **S3 Tables**:
- **customer_analytics** — monthly customer metrics (revenue, segments, geography)
- **realtime_metrics** — hourly event metrics (traffic sources, conversions, revenue)

In [13]:
run_glue_job(GOLD_JOB)

▶ Started ecommerce-analytics-gold-layer-dev (run: jr_5d7f592bf8870128ca9a54a956b767715487e49472e358640383c1264c16b907)
✅ ecommerce-analytics-gold-layer-dev SUCCEEDED (80s, run jr_5d7f592bf8870128ca9a54a956b767715487e49472e358640383c1264c16b907)


'SUCCEEDED'

## Step 10: Validate Gold Layer

Write Athena queries to validate the gold tables:
1. Show monthly customer analytics (analysis_month, total_customers, total_revenue, avg_order_value)
2. Show recent hourly metrics (metric_hour, total_events, unique_users, hourly_revenue)

Gold tables are on **S3 Tables** — pass `catalog=S3T_CATALOG, database=GOLD_DB`.

In [14]:
monthly = run_query("""
    SELECT analysis_month, total_customers, new_customers, returning_customers, total_orders,
           ROUND(total_revenue, 2) AS total_revenue, ROUND(avg_order_value, 2) AS avg_order_value,
           cities_served, states_served
    FROM customer_analytics
    ORDER BY analysis_month
""", catalog=S3T_CATALOG, database=GOLD_DB)
print(monthly.to_string(index=False))

                analysis_month total_customers new_customers returning_customers total_orders total_revenue avg_order_value cities_served states_served
2024-12-01 00:00:00.000000 UTC              35            34                  24          109      31664.73      290.500000            35            27
2025-01-01 00:00:00.000000 UTC              82            49                  51          144      37326.43      259.210000            82            46
2025-02-01 00:00:00.000000 UTC             103            42                  71          157      39507.88      251.640000           103            46
2025-03-01 00:00:00.000000 UTC             128            50                  89          169      43828.48      259.340000           126            52
2025-04-01 00:00:00.000000 UTC             132            55                  93          172      45978.67      267.320000           130            54
2025-05-01 00:00:00.000000 UTC             139            47                 101        

In [15]:
hourly = run_query("""
    SELECT metric_hour, total_events, unique_users, unique_sessions, page_views, purchases,
           ROUND(hourly_revenue, 2) AS hourly_revenue, google_traffic, facebook_traffic, direct_traffic
    FROM realtime_metrics
    ORDER BY metric_hour DESC
    LIMIT 10
""", catalog=S3T_CATALOG, database=GOLD_DB)
print(hourly.to_string(index=False))

# Revenue is the product price on purchase events; purchases of products deleted via CDC have no
# product match, so the most recent hours can show 0. The highest-revenue hours:
top_rev = run_query("""
    SELECT metric_hour, total_events, purchases, ROUND(hourly_revenue, 2) AS hourly_revenue,
           google_traffic, facebook_traffic, direct_traffic
    FROM realtime_metrics
    ORDER BY hourly_revenue DESC
    LIMIT 10
""", catalog=S3T_CATALOG, database=GOLD_DB)
print(top_rev.to_string(index=False))
nonzero = run_query("SELECT COUNT(*) AS hours_with_revenue FROM realtime_metrics WHERE hourly_revenue > 0",
                    catalog=S3T_CATALOG, database=GOLD_DB)
print(nonzero.to_string(index=False))


                   metric_hour total_events unique_users unique_sessions page_views purchases hourly_revenue google_traffic facebook_traffic direct_traffic
2026-09-30 07:00:00.000000 UTC            2            1               1          0         2           0.00              0                0              0
2026-09-30 02:00:00.000000 UTC            2            1               1          0         0           0.00              0                0              0
2026-09-29 12:00:00.000000 UTC            2            1               1          0         0           0.00              0                0              2
2026-09-28 21:00:00.000000 UTC            2            1               1          2         0           0.00              2                0              0
2026-09-27 14:00:00.000000 UTC            4            2               2          0         2           0.00              2                0              2
2026-09-25 02:00:00.000000 UTC            2            1        

---
# Part 5: Iceberg Features

## Step 11: Snapshot Inspection

Iceberg maintains a log of every change as immutable **snapshots**. Query the
`$snapshots` and `$history` metadata tables to see when data was written,
what operation occurred (append, overwrite), and how many files were added.

Use the bronze orders table for this — reference metadata tables as:
- `"{DATABASE}"."orders$snapshots"`
- `"{DATABASE}"."orders$history"`

In [16]:
baseline = run_query(f"SELECT COUNT(*) AS order_count, ROUND(SUM(total_amount), 2) AS total_revenue FROM {DATABASE}.orders")
snaps_before = run_query(f'SELECT snapshot_id, committed_at, operation FROM "{DATABASE}"."orders$snapshots" ORDER BY committed_at')
print('Baseline:'); print(baseline.to_string(index=False))
print('Snapshots before supplemental CDC:'); print(snaps_before.to_string(index=False))
hist_ts = snaps_before['committed_at'].iloc[-1]    # latest snapshot = state before the change
print('Historical timestamp for time travel:', hist_ts)

# 2. Apply supplemental CDC (3 order updates + 2 order deletes, plus users/products changes)
run_glue_job(CDC_JOB, arguments={'--supplemental_path': f's3://{BUCKET}/datasets/cdc_data/supplemental'})

# 3. Snapshot metadata
snapshots = run_query(f"""
    SELECT snapshot_id, committed_at, operation, summary
    FROM "{DATABASE}"."orders$snapshots"
    ORDER BY committed_at
""")
print(f'orders snapshots: {len(snapshots)}')
print(snapshots.to_string(index=False))

Baseline:
order_count total_revenue
       2133     556462.16
Snapshots before supplemental CDC:
        snapshot_id                committed_at operation
1402999689570794797 2026-11-12 13:09:47.076 UTC    append
8158321595349035171 2026-11-12 13:17:03.296 UTC overwrite
4812097465724720033 2026-11-12 16:26:34.033 UTC overwrite
Historical timestamp for time travel: 2026-11-12 16:26:34.033 UTC
▶ Started ecommerce-analytics-cdc-bronze-dev (run: jr_4ea642fd48dc0a7a7d02a388ef675d5ec56a776d30b8222037275b6708096776)
✅ ecommerce-analytics-cdc-bronze-dev SUCCEEDED (108s, run jr_4ea642fd48dc0a7a7d02a388ef675d5ec56a776d30b8222037275b6708096776)
orders snapshots: 4
        snapshot_id                committed_at operation                                                                                                                                                                                                                                                                                         

In [17]:
history = run_query(f"""
    SELECT made_current_at, snapshot_id, parent_id, is_current_ancestor
    FROM "{DATABASE}"."orders$history"
    ORDER BY made_current_at
""")
print(history.to_string(index=False))

            made_current_at         snapshot_id           parent_id is_current_ancestor
2026-11-12 13:09:47.076 UTC 1402999689570794797                None                true
2026-11-12 13:17:03.296 UTC 8158321595349035171 1402999689570794797                true
2026-11-12 16:26:34.033 UTC 4812097465724720033 8158321595349035171                true
2026-11-12 16:35:02.159 UTC  720563117546524448 4812097465724720033                true


## Step 12: Time-Travel Query

`FOR TIMESTAMP AS OF` reads the table as it existed at a previous point in time. We compare the
**bronze orders** table at the timestamp captured in Step 11 (before the supplemental CDC was applied)
with its current state. The difference comes from the supplemental updates and deletes applied
through `MERGE INTO`.

> Note: the events job is append-only, so it is deliberately **not** used for time travel.


In [18]:
current = run_query(f"SELECT COUNT(*) AS order_count, ROUND(SUM(total_amount), 2) AS total_revenue FROM {DATABASE}.orders")
historical = run_query(f"""
    SELECT COUNT(*) AS order_count, ROUND(SUM(total_amount), 2) AS total_revenue
    FROM {DATABASE}.orders FOR TIMESTAMP AS OF TIMESTAMP '{hist_ts}'
""")
comparison = pd.DataFrame({
    'metric': ['Order count', 'Total revenue'],
    'historical': historical.iloc[0].tolist(),
    'current': current.iloc[0].tolist(),
})
print(f'Time travel as of {hist_ts}')
print(comparison.to_string(index=False))
assert comparison['historical'].tolist() != comparison['current'].tolist(), 'Historical and current state should differ'

Time travel as of 2026-11-12 16:26:34.033 UTC
       metric historical   current
  Order count       2133      2131
Total revenue  556462.16 555744.83


---
## Step 13: Medallion Architecture Summary

Write a summary cell that queries all three layers and prints row counts.
Bronze uses the default catalog; Silver and Gold use `S3T_CATALOG`.

In [19]:
summary_rows = []
def table_count(layer, table, **kw):
    try:
        ref = f'{DATABASE}.{table}' if layer == 'Bronze' else table
        n = int(run_query(f'SELECT COUNT(*) AS c FROM {ref}', **kw).iloc[0]['c'])
        summary_rows.append((layer, table, n, 'OK' if n > 0 else 'EMPTY'))
    except Exception as e:
        summary_rows.append((layer, table, None, f'ERROR: {str(e)[:60]}'))

for t in ['orders', 'users', 'products', 'events']:
    table_count('Bronze', t)
for t in ['order_details', 'enriched_events', 'product_performance']:
    table_count('Silver', t, catalog=S3T_CATALOG, database=SILVER_DB)
for t in ['customer_analytics', 'realtime_metrics']:
    table_count('Gold', t, catalog=S3T_CATALOG, database=GOLD_DB)

summary = pd.DataFrame(summary_rows, columns=['Layer', 'Table', 'Row Count', 'Status'])
print(summary.to_string(index=False))
print()
print('Latest Glue job runs (from the Glue API):')
for job in (CDC_JOB, EVENTS_JOB, SILVER_JOB, GOLD_JOB):
    for r in glue.get_job_runs(JobName=job, MaxResults=3)['JobRuns'][::-1]:
        print(f"  {job}: {r['JobRunState']} ({r.get('ExecutionTime', 0)}s) run {r['Id']}")


 Layer               Table  Row Count Status
Bronze              orders       2131     OK
Bronze               users        912     OK
Bronze            products        393     OK
Bronze              events        996     OK
Silver       order_details       2024     OK
Silver     enriched_events        996     OK
Silver product_performance        393     OK
  Gold  customer_analytics         14     OK
  Gold    realtime_metrics        491     OK

Latest Glue job runs (from the Glue API):
  ecommerce-analytics-cdc-bronze-dev: SUCCEEDED (111s) run jr_88e8dc2856926993ea5fcf9b3e7b0687887267e0aa505069c1d576e689150253
  ecommerce-analytics-cdc-bronze-dev: SUCCEEDED (175s) run jr_764188bd21bc9c6f0c71ccb4f8cde48064d086b3fd845731c3569621c1d35791
  ecommerce-analytics-cdc-bronze-dev: SUCCEEDED (108s) run jr_4ea642fd48dc0a7a7d02a388ef675d5ec56a776d30b8222037275b6708096776
  ecommerce-analytics-events-bronze-dev: SUCCEEDED (55s) run jr_fbbca8552a361e3f7b274d01728e01d7e0686cd5ba58146d401df2d95c3230

---
## Run notes

- **Bronze `orders` history:** this run replayed the original CDC (a snapshot with 2133 rows) before the
  supplemental CDC was applied (2131 rows). The earlier snapshots in `$snapshots` come from a previous run
  of the same job against the same table.
- **Events are duplicated 2x (996 rows = 2 x 498):** the events job is append-only and was run a second
  time. Silver `enriched_events` and the Gold `realtime_metrics` volumes therefore count each event twice;
  ratios such as conversion rates are unaffected.
- **Order of builds:** Silver and Gold were built before the supplemental CDC, so they reflect the
  pre-supplemental state (supplemental updates move two orders into 2026-12 in `customer_analytics`).
- **Source data quality:** 5 orders have an invalid date (2025-02-29), excluded from the monthly Gold aggregation.
